In [3]:
import io
import zipfile
import requests
import frontmatter


def _download_repo_zip(repo_owner, repo_name):
    """Download repository zip using GitHub default branch automatically."""
    url = f"https://api.github.com/repos/{repo_owner}/{repo_name}/zipball"
    headers = {"User-Agent": "aihero-course-notebook"}
    resp = requests.get(url, headers=headers, timeout=60)

    if resp.status_code != 200:
        raise Exception(
            f"Failed to download repository zip for {repo_owner}/{repo_name}: "
            f"HTTP {resp.status_code}"
        )

    return resp


def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.

    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name

    Returns:
        List of dictionaries containing file content and metadata
    """
    resp = _download_repo_zip(repo_owner, repo_name)

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))

    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith(".md") or filename_lower.endswith(".mdx")):
            continue

        with zf.open(file_info) as f_in:
            content = f_in.read().decode("utf-8", errors="ignore")

        # Some repos contain invalid YAML frontmatter.
        # Fallback to raw content so ingestion does not fail.
        try:
            post = frontmatter.loads(content)
            data = post.to_dict()
        except Exception as e:
            data = {
                "content": content,
                "frontmatter_error": str(e),
            }

        data["filename"] = filename
        repository_data.append(data)

    zf.close()
    return repository_data

In [ ]:
dtc_langchain = read_repo_data('langchain-ai', 'langchain')
dtc_microsoft = read_repo_data('microsoft', 'semantic-kernel')
dtc_openai = read_repo_data('openai', 'openai-cookbook')
print(f"Langchain documents: {len(dtc_langchain)}")
print(f"microsoft documents: {len(dtc_microsoft)}")
print(f"openai documents: {len(dtc_openai)}")

Error processing microsoft-semantic-kernel-83ff0a5/docs/decisions/adr-short-template.md: while parsing a flow mapping
  in "<unicode string>", line 3, column 9
did not find expected ',' or '}'
  in "<unicode string>", line 3, column 74
Error processing microsoft-semantic-kernel-83ff0a5/docs/decisions/adr-template.md: while parsing a flow mapping
  in "<unicode string>", line 3, column 9
did not find expected ',' or '}'
  in "<unicode string>", line 3, column 74


In [ ]:
langchain-ai/langchain
microsoft/semantic-kernel
openai/openai-cookbook